In [1]:
import os

In [2]:
%pwd

'c:\\Users\\sagal\\OneDrive\\Desktop\\Let us build\\Linear_regression_01\\research'

In [3]:
os.chdir('../')

In [4]:
%pwd

'c:\\Users\\sagal\\OneDrive\\Desktop\\Let us build\\Linear_regression_01'

In [5]:
from dataclasses import dataclass
from pathlib import Path
from Linear_regression_01.logging import logger

In [6]:
#entity

@dataclass(frozen=True)
class DataTransformationConfig:
    root_dir: Path
    data_path: Path
    STATUS_FILE: Path

In [7]:
from Linear_regression_01 import *
from Linear_regression_01.utils.common import read_yaml,create_directories

In [8]:
# Ensure your constant paths are defined
CONFIG_FILE_PATH = Path("config/config.yaml")
PARAMS_FILE_PATH = Path("params.yaml")
SCHEMA_FILE_PATH = Path("schema.yaml")

In [15]:
#configuration manager
class configurationManager:
    def __init__(
        self,
        config_filepath = CONFIG_FILE_PATH,     # Access to constants
        params_filepath = PARAMS_FILE_PATH,
        schema_filepath = SCHEMA_FILE_PATH):

        self.config = read_yaml(config_filepath) # read all config and params yaml files
        self.params = read_yaml(params_filepath)
        self.schema = read_yaml(schema_filepath)

        create_directories([self.config.artifacts_root])
    
    def get_data_transformation_config(self) -> DataTransformationConfig:
        config = self.config.data_transformation
        schema = self.schema.COLUMNS

        create_directories([config.root_dir])

        data_transformation_config = DataTransformationConfig(
            root_dir = config.root_dir,
            STATUS_FILE = config.STATUS_FILE,
            data_path = config.data_path,
        )

        return data_transformation_config

In [10]:
import os
import urllib.request as request
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression   
import tarfile #changes on the basis of the file type if zip use import zipfile
from Linear_regression_01.utils.common import get_size

In [17]:
#comppnents #main thing

# imagine the flow of the code first check valid -> then transform(split and train)
class DataTransformation:
    def __init__(self, data_transformation_config):
        self.config = data_transformation_config
    
    def check_validation_status(self) -> bool:
        with open(self.config.STATUS_FILE, "r") as f:
            status = f.read().split(" ")[-1].strip()
            
        if status == "True":
            return True
        else:
            return False
    
    def initiate_data_transformation(self):
        #check the schema validation from prev functon
        if self.check_validation_status():
            #read raw data (stg_1)
            df = pd.read_csv(self.config.data_path)
            
            #split data
            train, test = train_test_split(df, test_size=0.2, random_state=42)
            
            #save data files to path
            train.to_csv(os.path.join(self.config.root_dir, "train.csv"), index=False)
            test.to_csv(os.path.join(self.config.root_dir, "test.csv"), index=False)
            
            print("Data split into training and testing sets successfully.")
            print(f"Train shape: {train.shape}, Test shape: {test.shape}")
            
        else:
            raise Exception("Data Validation Stage failed! Cannot proceed with Data Transformation.")

In [19]:
#pipeline

try:
    config = configurationManager()
    data_transformation_config = config.get_data_transformation_config()
    data_transformation = DataTransformation(data_transformation_config)
    
    #run chek 1
    validation_true = data_transformation.check_validation_status()

    if validation_true:
        data_transformation.initiate_data_transformation()
        print("Data Transformation stage completed successfully")
    else:
        print("Data Transformation stage failed: recheck")

except Exception as e:
    raise e



[2026-06-11 18:01:58,797: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-06-11 18:01:58,801: INFO: common: yaml file: params.yaml loaded successfully]
[2026-06-11 18:01:58,806: INFO: common: yaml file: schema.yaml loaded successfully]
[2026-06-11 18:01:58,809: INFO: common: created directory at artifacts]
[2026-06-11 18:01:58,812: INFO: common: created directory at artifacts/data_transformation]
Data split into training and testing sets successfully.
Train shape: (16512, 10), Test shape: (4128, 10)
Data Transformation stage completed successfully
